In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import networkx as nx
import seaborn as sns

In [ ]:

file_path = "../data/train.txt"
triples = []

with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:  # skip empty lines
            head, relation, tail = line.split()
            triples.append((head, relation, tail))
print(f"Total triples loaded: {len(triples)}")

In [ ]:
df = pd.DataFrame(triples, columns=["head", "relation", "tail"])
entities = pd.unique(df[["head", "tail"]].values.ravel())
relations = pd.unique(df["relation"].values.ravel())
unique_nodes = pd.unique(df[["head", "tail"]].values.ravel())
unique_nodes = unique_nodes.tolist()
relation_counts = df["relation"].value_counts()
relation_count_dict = relation_counts.to_dict()
relation_stats_df = relation_counts.reset_index()
relation_stats_df.columns = ["relation", "count"]


In [ ]:
G = nx.MultiDiGraph()
for _, row in df.iterrows():
    G.add_edge(
        row["head"],
        row["tail"],
        relation=row["relation"]
    )
print(f"Graph has {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.")

## Step 1: Examine All Relationships in the Dataset

First, let's understand what relationships exist and their frequencies.

In [ ]:
# Display all unique relations and their counts
print("ALL RELATIONSHIPS IN THE METAFAM KNOWLEDGE GRAPH")
print(f"\nTotal unique relations: {len(relations)}")
print(f"Total triples: {len(df)}")
print(f"\nRelationship distribution:\n")

# Sort by count for better understanding
relation_stats_sorted = relation_stats_df.sort_values('count', ascending=False)
for idx, row in relation_stats_sorted.iterrows():
    print(f"  {row['relation']:30s} : {row['count']:4d} instances")

print("\n")

## Step 2: Build Rule Mining Infrastructure

Create functions to discover and validate logical rules in the knowledge graph.

In [ ]:
# Create dictionaries for efficient rule mining
# Format: {(head, relation): [list of tails]}
relation_dict = {}
reverse_dict = {}  # For inverse lookups: {(tail, relation): [list of heads]}

for _, row in df.iterrows():
    head, relation, tail = row['head'], row['relation'], row['tail']
    
    # Forward direction
    key = (head, relation)
    if key not in relation_dict:
        relation_dict[key] = []
    relation_dict[key].append(tail)
    
    # Reverse direction
    rev_key = (tail, relation)
    if rev_key not in reverse_dict:
        reverse_dict[rev_key] = []
    reverse_dict[rev_key].append(head)

print(f"Built relation dictionaries:")
print(f"  Forward entries: {len(relation_dict)}")
print(f"  Reverse entries: {len(reverse_dict)}")

In [ ]:
class RuleMiner:
    """Class for discovering and validating logical rules in knowledge graphs"""
    
    def __init__(self, df, G):
        self.df = df
        self.G = G
        self.relation_dict = relation_dict
        self.reverse_dict = reverse_dict
    
    def mine_horn_clause_2hop(self, rel1, rel2, rel3, verbose=False):
        """
        Mine rule: rel1(X,Y) ∧ rel2(Y,Z) → rel3(X,Z)
        
        Returns:
            support: number of instances where rule holds
            confidence: fraction of times rule is correct
            examples: list of concrete examples
        """
        support = 0
        total_candidates = 0
        examples = []
        
        # Find all instances of rel1(X,Y)
        for (head, relation), tails in self.relation_dict.items():
            if relation == rel1:
                for y in tails:
                    # Find all instances of rel2(Y,Z)
                    if (y, rel2) in self.relation_dict:
                        for z in self.relation_dict[(y, rel2)]:
                            total_candidates += 1
                            
                            # Check if rel3(X,Z) exists
                            if (head, rel3) in self.relation_dict and z in self.relation_dict[(head, rel3)]:
                                support += 1
                                if len(examples) < 5:  # Collect up to 5 examples
                                    examples.append((head, y, z))
        
        confidence = support / total_candidates if total_candidates > 0 else 0
        
        return {
            'rule': f"{rel1}(X,Y) ∧ {rel2}(Y,Z) → {rel3}(X,Z)",
            'support': support,
            'total_candidates': total_candidates,
            'confidence': confidence,
            'examples': examples
        }
    
    def mine_inverse_rule(self, rel1, rel2, verbose=False):
        """
        Mine rule: rel1(X,Y) → rel2(Y,X)
        
        Returns:
            support: number of instances where rule holds
            confidence: fraction of times rule is correct
            examples: list of concrete examples
        """
        support = 0
        total_rel1 = 0
        examples = []
        
        # Find all instances of rel1(X,Y)
        for (head, relation), tails in self.relation_dict.items():
            if relation == rel1:
                for tail in tails:
                    total_rel1 += 1
                    
                    # Check if rel2(tail, head) exists (inverse)
                    if (tail, rel2) in self.relation_dict and head in self.relation_dict[(tail, rel2)]:
                        support += 1
                        if len(examples) < 5:
                            examples.append((head, tail))
        
        confidence = support / total_rel1 if total_rel1 > 0 else 0
        
        return {
            'rule': f"{rel1}(X,Y) → {rel2}(Y,X)",
            'support': support,
            'total_candidates': total_rel1,
            'confidence': confidence,
            'examples': examples
        }
    
    def mine_horn_clause_3hop(self, rel1, rel2, rel3, rel4, verbose=False):
        """
        Mine rule: rel1(X,Y) ∧ rel2(Y,Z) ∧ rel3(Z,W) → rel4(X,W)
        
        Returns:
            support: number of instances where rule holds
            confidence: fraction of times rule is correct
            examples: list of concrete examples
        """
        support = 0
        total_candidates = 0
        examples = []
        
        # Find all instances of rel1(X,Y)
        for (head, relation), ys in self.relation_dict.items():
            if relation == rel1:
                for y in ys:
                    # Find all instances of rel2(Y,Z)
                    if (y, rel2) in self.relation_dict:
                        for z in self.relation_dict[(y, rel2)]:
                            # Find all instances of rel3(Z,W)
                            if (z, rel3) in self.relation_dict:
                                for w in self.relation_dict[(z, rel3)]:
                                    total_candidates += 1
                                    
                                    # Check if rel4(X,W) exists
                                    if (head, rel4) in self.relation_dict and w in self.relation_dict[(head, rel4)]:
                                        support += 1
                                        if len(examples) < 5:
                                            examples.append((head, y, z, w))
        
        confidence = support / total_candidates if total_candidates > 0 else 0
        
        return {
            'rule': f"{rel1}(X,Y) ∧ {rel2}(Y,Z) ∧ {rel3}(Z,W) → {rel4}(X,W)",
            'support': support,
            'total_candidates': total_candidates,
            'confidence': confidence,
            'examples': examples
        }

print("RuleMiner class created successfully!")
miner = RuleMiner(df, G)
print("RuleMiner instance initialized.")

## Step 3: Discover Logical Rules

Now let's discover at least 5 logical rules by analyzing relationship compositions.

### Rule 1: Parent-Grandparent Composition (Horn Clause)

**Rule:** motherOf(X,Y) ∧ motherOf(Y,Z) → grandmotherOf(X,Z)

**Intuition:** If X is the mother of Y, and Y is the mother of Z, then X is the grandmother of Z.

In [ ]:
# Rule 1: motherOf(X,Y) ∧ motherOf(Y,Z) → grandmotherOf(X,Z)
rule1 = miner.mine_horn_clause_2hop('motherOf', 'motherOf', 'grandmotherOf')


print("RULE 1: motherOf(X,Y) ∧ motherOf(Y,Z) → grandmotherOf(X,Z)")

print(f"\n Metrics:")
print(f"   Support: {rule1['support']} instances")
print(f"   Total Candidates: {rule1['total_candidates']}")
print(f"   Confidence: {rule1['confidence']:.2%}")

print(f"\n Concrete Examples:")
for i, (x, y, z) in enumerate(rule1['examples'], 1):
    print(f"   {i}. {x} --motherOf--> {y} --motherOf--> {z}")
    print(f"      ✓ Verified: {x} --grandmotherOf--> {z}")


### Rule 2: Father-Grandfather Composition (Horn Clause)

**Rule:** fatherOf(X,Y) ∧ fatherOf(Y,Z) → grandfatherOf(X,Z)

**Intuition:** If X is the father of Y, and Y is the father of Z, then X is the grandfather of Z.

In [ ]:
# Rule 2: fatherOf(X,Y) ∧ fatherOf(Y,Z) → grandfatherOf(X,Z)
rule2 = miner.mine_horn_clause_2hop('fatherOf', 'fatherOf', 'grandfatherOf')

print("RULE 2: fatherOf(X,Y) ∧ fatherOf(Y,Z) → grandfatherOf(X,Z)")
print(f"\nMetrics:")
print(f"   Support: {rule2['support']} instances")
print(f"   Total Candidates: {rule2['total_candidates']}")
print(f"   Confidence: {rule2['confidence']:.2%}")

print(f"\nConcrete Examples:")
for i, (x, y, z) in enumerate(rule2['examples'], 1):
    print(f"   {i}. {x} --fatherOf--> {y} --fatherOf--> {z}")
    print(f"      ✓ Verified: {x} --grandfatherOf--> {z}")

### Rule 3: Cross-Gender Grandmother Composition (Horn Clause)

**Rule:** motherOf(X,Y) ∧ fatherOf(Y,Z) → grandmotherOf(X,Z)

**Intuition:** If X is the mother of Y, and Y is the father of Z, then X is the grandmother of Z.

In [ ]:
# Rule 3: motherOf(X,Y) ∧ fatherOf(Y,Z) → grandmotherOf(X,Z)
rule3 = miner.mine_horn_clause_2hop('motherOf', 'fatherOf', 'grandmotherOf')


print("RULE 3: motherOf(X,Y) ∧ fatherOf(Y,Z) → grandmotherOf(X,Z)")
print(f"\n Metrics:")
print(f"   Support: {rule3['support']} instances")
print(f"   Total Candidates: {rule3['total_candidates']}")
print(f"   Confidence: {rule3['confidence']:.2%}")

print(f"\n Concrete Examples:")
for i, (x, y, z) in enumerate(rule3['examples'], 1):
    print(f"   {i}. {x} --motherOf--> {y} --fatherOf--> {z}")
    print(f"      ✓ Verified: {x} --grandmotherOf--> {z}")


### Rule 4: Mother-Daughter/Son Inverse (Inverse Rule)

**Rule:** motherOf(X,Y) → daughterOf(Y,X) OR sonOf(Y,X)

**Intuition:** If X is the mother of Y, then Y is either the daughter or son of X. This is a bidirectional relationship.

**Note:** We'll check either daughterOf or sonOf as the inverse.

In [ ]:
# Rule 4: motherOf(X,Y) → daughterOf(Y,X) OR sonOf(Y,X)
# Since there are two possible inverses, we'll check both
rule4a = miner.mine_inverse_rule('motherOf', 'daughterOf')
rule4b = miner.mine_inverse_rule('motherOf', 'sonOf')

# Combine results
total_motherOf = rule4a['total_candidates']
support_combined_rule4 = rule4a['support'] + rule4b['support']
confidence_combined_rule4 = support_combined_rule4 / total_motherOf if total_motherOf > 0 else 0


print("RULE 4: motherOf(X,Y) → daughterOf(Y,X) OR sonOf(Y,X)")

print(f"\n Metrics:")
print(f"   Total motherOf instances: {total_motherOf}")
print(f"   Support (daughterOf): {rule4a['support']} instances")
print(f"   Support (sonOf): {rule4b['support']} instances")
print(f"   Combined Support: {support_combined_rule4} instances")
print(f"   Combined Confidence: {confidence_combined_rule4:.2%}")

print(f"\n Concrete Examples (daughterOf):")
for i, (x, y) in enumerate(rule4a['examples'][:3], 1):
    print(f"   {i}. {x} --motherOf--> {y}")
    print(f"      ✓ Verified: {y} --daughterOf--> {x}")

print(f"\n Concrete Examples (sonOf):")
for i, (x, y) in enumerate(rule4b['examples'][:3], 1):
    print(f"   {i}. {x} --motherOf--> {y}")
    print(f"      ✓ Verified: {y} --sonOf--> {x}")

### Rule 5: Sibling Symmetry (Inverse Rule)

**Rule:** sisterOf(X,Y) → sisterOf(Y,X) OR brotherOf(Y,X)

**Intuition:** If X is the sister of Y, then Y is either the sister or brother of X. Sibling relationships are symmetric (bidirectional).

In [ ]:
# Rule 5: sisterOf(X,Y) → sisterOf(Y,X) OR brotherOf(Y,X)
rule5a = miner.mine_inverse_rule('sisterOf', 'sisterOf')
rule5b = miner.mine_inverse_rule('sisterOf', 'brotherOf')

# Combine results
total_sisterOf = rule5a['total_candidates']
support_combined_rule5 = rule5a['support'] + rule5b['support']
confidence_combined_rule5 = support_combined_rule5 / total_sisterOf if total_sisterOf > 0 else 0

print("RULE 5: sisterOf(X,Y) → sisterOf(Y,X) OR brotherOf(Y,X)")
print(f"\n Metrics:")
print(f"   Total sisterOf instances: {total_sisterOf}")
print(f"   Support (sisterOf inverse): {rule5a['support']} instances")
print(f"   Support (brotherOf): {rule5b['support']} instances")
print(f"   Combined Support: {support_combined_rule5} instances")
print(f"   Combined Confidence: {confidence_combined_rule5:.2%}")

print(f"\n Concrete Examples (sisterOf inverse):")
for i, (x, y) in enumerate(rule5a['examples'][:3], 1):
    print(f"   {i}. {x} --sisterOf--> {y}")
    print(f"      ✓ Verified: {y} --sisterOf--> {x}")

print(f"\n Concrete Examples (brotherOf):")
for i, (x, y) in enumerate(rule5b['examples'][:3], 1):
    print(f"   {i}. {x} --sisterOf--> {y}")
    print(f"      ✓ Verified: {y} --brotherOf--> {x}")

### Rule 6: Great-Grandmother Composition (Complex 3-hop Pattern)

**Rule:** motherOf(X,Y) ∧ motherOf(Y,Z) ∧ motherOf(Z,W) → greatGrandmotherOf(X,W)

**Intuition:** If X is the mother of Y, Y is the mother of Z, and Z is the mother of W, then X is the great-grandmother of W. This is a 3-generation rule.

In [ ]:
# Rule 6: motherOf(X,Y) ∧ motherOf(Y,Z) ∧ motherOf(Z,W) → greatGrandmotherOf(X,W)
rule6 = miner.mine_horn_clause_3hop('motherOf', 'motherOf', 'motherOf', 'greatGrandmotherOf')


print("RULE 6: motherOf(X,Y) ∧ motherOf(Y,Z) ∧ motherOf(Z,W) → greatGrandmotherOf(X,W)")
print(f"\n Metrics:")
print(f"   Support: {rule6['support']} instances")
print(f"   Total Candidates: {rule6['total_candidates']}")
print(f"   Confidence: {rule6['confidence']:.2%}")

print(f"\n Concrete Examples:")
for i, (x, y, z, w) in enumerate(rule6['examples'], 1):
    print(f"   {i}. {x} --motherOf--> {y} --motherOf--> {z} --motherOf--> {w}")
    print(f"      ✓ Verified: {x} --greatGrandmotherOf--> {w}")

### Rule 7: Cross-Gender Grandfather Composition (Horn Clause)

**Rule:** fatherOf(X,Y) ∧ motherOf(Y,Z) → grandfatherOf(X,Z)

**Intuition:** If X is the father of Y, and Y is the mother of Z, then X is the grandfather of Z.

In [ ]:
# Rule 7: fatherOf(X,Y) ∧ motherOf(Y,Z) → grandfatherOf(X,Z)
rule7 = miner.mine_horn_clause_2hop('fatherOf', 'motherOf', 'grandfatherOf')

print("RULE 7: fatherOf(X,Y) ∧ motherOf(Y,Z) → grandfatherOf(X,Z)")
print(f"\n Metrics:")
print(f"   Support: {rule7['support']} instances")
print(f"   Total Candidates: {rule7['total_candidates']}")
print(f"   Confidence: {rule7['confidence']:.2%}")

print(f"\n Concrete Examples:")
for i, (x, y, z) in enumerate(rule7['examples'], 1):
    print(f"   {i}. {x} --fatherOf--> {y} --motherOf--> {z}")
    print(f"      ✓ Verified: {x} --grandfatherOf--> {z}")

## Step 4: Summary of Discovered Rules

Let's create a comprehensive summary table of all discovered rules.

In [ ]:
# Create summary table
summary_data = []

# Rule 1
summary_data.append({
    'Rule #': 1,
    'Type': 'Horn Clause (2-hop)',
    'Rule': rule1['rule'],
    'Support': rule1['support'],
    'Total Candidates': rule1['total_candidates'],
    'Confidence': f"{rule1['confidence']:.2%}"
})

# Rule 2
summary_data.append({
    'Rule #': 2,
    'Type': 'Horn Clause (2-hop)',
    'Rule': rule2['rule'],
    'Support': rule2['support'],
    'Total Candidates': rule2['total_candidates'],
    'Confidence': f"{rule2['confidence']:.2%}"
})

# Rule 3
summary_data.append({
    'Rule #': 3,
    'Type': 'Horn Clause (2-hop)',
    'Rule': rule3['rule'],
    'Support': rule3['support'],
    'Total Candidates': rule3['total_candidates'],
    'Confidence': f"{rule3['confidence']:.2%}"
})

# Rule 4 (combined)
summary_data.append({
    'Rule #': 4,
    'Type': 'Inverse Rule',
    'Rule': 'motherOf(X,Y) → daughterOf(Y,X) OR sonOf(Y,X)',
    'Support': support_combined_rule4,
    'Total Candidates': total_motherOf,
    'Confidence': f"{confidence_combined_rule4:.2%}"
})

# Rule 5 (combined)
summary_data.append({
    'Rule #': 5,
    'Type': 'Inverse Rule',
    'Rule': 'sisterOf(X,Y) → sisterOf(Y,X) OR brotherOf(Y,X)',
    'Support': support_combined_rule5,
    'Total Candidates': total_sisterOf,
    'Confidence': f"{confidence_combined_rule5:.2%}"
})

# Rule 6
summary_data.append({
    'Rule #': 6,
    'Type': 'Complex (3-hop)',
    'Rule': rule6['rule'],
    'Support': rule6['support'],
    'Total Candidates': rule6['total_candidates'],
    'Confidence': f"{rule6['confidence']:.2%}"
})

# Rule 7
summary_data.append({
    'Rule #': 7,
    'Type': 'Horn Clause (2-hop)',
    'Rule': rule7['rule'],
    'Support': rule7['support'],
    'Total Candidates': rule7['total_candidates'],
    'Confidence': f"{rule7['confidence']:.2%}"
})

summary_df = pd.DataFrame(summary_data)

print("COMPREHENSIVE SUMMARY OF ALL DISCOVERED RULES")

print(summary_df.to_string(index=False))

## Step 5: Visualize Rule Metrics

Let's create visualizations to better understand the discovered rules.

In [ ]:
# Extract numeric confidence values (needed for all plots)
confidences = []
for conf_str in summary_df['Confidence']:
    confidences.append(float(conf_str.strip('%')) / 100)

summary_df['Confidence_Numeric'] = confidences

# Color scheme
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2']

In [ ]:
# PLOT 1: Confidence by Rule
plt.figure(figsize=(10, 6))

plt.bar(
    summary_df['Rule #'],
    summary_df['Confidence_Numeric'],
    color=colors,
    alpha=0.7,
    edgecolor='black'
)
plt.xlabel('Rule Number', fontsize=12)
plt.ylabel('Confidence', fontsize=12)
plt.title('Rule Confidence Scores\n(How often the rule holds)', fontsize=13, fontweight='bold')
plt.ylim(0, 1.1)
plt.axhline(y=1.0, color='green', linestyle='--', alpha=0.5, label='100% confidence')
plt.axhline(y=0.8, color='orange', linestyle='--', alpha=0.5, label='80% threshold')
plt.grid(axis='y', alpha=0.3)
plt.legend()

# Add percentage labels on bars
for i, (rule_num, conf) in enumerate(zip(summary_df['Rule #'], summary_df['Confidence_Numeric'])):
    plt.text(rule_num, conf + 0.02, f'{conf:.0%}', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

print("Confidence Score Analysis:")
print("  6 out of 7 rules achieve 100% confidence!")

In [ ]:
# PLOT 2: Support by Rule
plt.figure(figsize=(10, 6))

plt.bar(
    summary_df['Rule #'],
    summary_df['Support'],
    color=colors,
    alpha=0.7,
    edgecolor='black'
)
plt.xlabel('Rule Number', fontsize=12)
plt.ylabel('Support (# instances)', fontsize=12)
plt.title('Rule Support\n(How many times the rule appears)', fontsize=13, fontweight='bold')
plt.grid(axis='y', alpha=0.3)

# Add count labels on bars
for rule_num, support in zip(summary_df['Rule #'], summary_df['Support']):
    plt.text(rule_num, support + 10, str(support), ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

print("Support Count Analysis:")
print("  Shows frequency of each rule pattern in the data")

In [ ]:
# PLOT 3: Rule Types Distribution
plt.figure(figsize=(8, 8))

type_counts = summary_df['Type'].value_counts()
colors_pie = ['#1f77b4', '#ff7f0e', '#2ca02c']

plt.pie(
    type_counts.values,
    labels=type_counts.index,
    autopct='%1.0f%%',
    colors=colors_pie,
    startangle=90,
    explode=[0.05] * len(type_counts),
    textprops={'fontsize': 11, 'fontweight': 'bold'}
)
plt.title('Distribution of Rule Types', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

print("Rule Type Distribution:")
print("  Mix of Horn clauses, inverse rules, and complex patterns")